In [4]:
! pip install jedi>=0.16
! pip install supervision
! pip install -q git+https://github.com/Deci-AI/super-gradients.git

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 181.5/181.5 kB 3.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
! sed -i 's/sghub.deci.ai/sg-hub-nv.s3.amazonaws.com/' /usr/local/lib/python3.11/dist-packages/super_gradients/training/pretrained_models.py
! sed -i 's/sghub.deci.ai/sg-hub-nv.s3.amazonaws.com/' /usr/local/lib/python3.11/dist-packages/super_gradients/training/utils/checkpoint_utils.py

In [6]:
from super_gradients.common.object_names import Models
from super_gradients.conversion import DetectionOutputFormatMode
from super_gradients.training import models
import torch
from super_gradients.training import Trainer
from super_gradients.training.losses import PPYoloELoss
from super_gradients.training.metrics import DetectionMetrics_050
from super_gradients.training.models.detection_models.pp_yolo_e import PPYoloEPostPredictionCallback
import supervision as sv

ds = sv.DetectionDataset.from_coco(
    images_directory_path=f"/content/test",
    annotations_path=f"/content/test/_ans.json",
    force_masks=False
)

DEVICE = 'cuda' if torch.cuda.is_available() else "cpu"
MODEL_ARCH = 'yolo_nas_s'
trainer = Trainer(experiment_name='CustomTest', ckpt_root_dir="/content/models/")
best_model = models.get(
    MODEL_ARCH,
    num_classes=5,
    checkpoint_path=f"/content/models/average_model.pth"
).to(DEVICE)
trainer.test(
    model=best_model,
    test_loader=test_data,
    test_metrics_list=DetectionMetrics_050(
        score_thres=0.1,
        top_k_predictions=300,
        num_cls=5,
        normalize_targets=True,
        post_prediction_callback=PPYoloEPostPredictionCallback(
            score_threshold=0.01,
            nms_top_k=1000,
            max_predictions=300,
            nms_threshold=0.7
        )
    )
)

FileNotFoundError: [Errno 2] No such file or directory: '/content/test/_coco_merged.json'

In [ ]:
# export the model for compatibility with Frigate

model.export("yolo_nas_s.onnx",
             output_predictions_format=DetectionOutputFormatMode.FLAT_FORMAT,
             max_predictions_per_image=20,
             num_pre_nms_predictions=300,
             confidence_threshold=0.4,
             input_image_shape=(320,320),
            )

In [ ]:
from google.colab import files

files.download('yolo_nas_s.onnx')